# Colab / Local Benchmark Runner

Single-GPU benchmark entrypoint for Google Colab and local Jupyter.

Flow:
1. Detect Colab vs local notebook
2. Mount Drive when running in Colab
3. Clone/update the repo into `/content` in Colab, or reuse the current local checkout
3. Install `bench/` dependencies
4. Build or reuse `llama-server`
5. Ensure model/mmproj artifacts
6. Run a smoke test
7. Run the requested benchmark matrix with resume support


In [ ]:
REPO_URL = "https://github.com/Chedrian07/TQ_LLAMACPP_VSION_BENCH.git"
BRANCH = "main"
LLAMA_CPP_REPO_URL = "https://github.com/Chedrian07/llama.cpp.git"
LLAMA_CPP_BRANCH = "master"
LLAMA_CPP_COMMIT = "685739763fa79038bb3b138c8cbc1e7f90ff7eaa"
WORKSPACE_ROOT = "/content"
DRIVE_ROOT = "/content/drive/MyDrive/tq_vlm_bench"

# User-editable run_bench.py defaults
# Valid model IDs: qwen3_vl_2b_instruct, qwen3_vl_2b_thinking
MODEL_ID = "qwen3_vl_2b_instruct"
MODEL_QUANT = "bf16"
# Optional extra model downloads: [("qwen3_vl_2b_thinking", "bf16")]
EXTRA_MODEL_DOWNLOADS = []
# Runtime groups are supported directly by run_bench.py, e.g. "core", ["core", "prod"], or "all".
RUNTIMES = "all"
# Benchmark groups are supported directly by run_bench.py, e.g. "p0", ["ai2d"], or "all".
BENCHMARKS = "all"
# Smoke defaults are kept small so you can sanity-check before the final run.
SMOKE_BENCHMARKS = ["ai2d"]
N_SAMPLES = -1
# Default to 4-way request parallelism for the final full benchmark run.
PARALLEL_OVERRIDE = 4
# Append arbitrary run_bench.py flags here, e.g. ["--seed", "123"].
EXTRA_RUN_BENCH_ARGS = []
RUN_KV_DUMP = True

# User-editable llama.cpp build defaults
LLAMA_BUILD_DIR_NAME = "build-colab"
LLAMA_BUILD_TYPE = "Release"
LLAMA_CUDA_ARCH_OVERRIDE = None   # e.g. "120"
LLAMA_EXTRA_CMAKE_ARGS = []       # e.g. ["-DGGML_CUDA_FA=OFF"]
LLAMA_BUILD_TARGETS = ["llama-server", "llama-kv-dump"]
LLAMA_BUILD_JOBS = None           # e.g. 8

RESULTS_TAG = f"colab_{MODEL_ID}_{MODEL_QUANT}"
RESUME = True
FORCE_REBUILD_SERVER = False
FORCE_REDOWNLOAD_MODEL = False

In [ ]:
import subprocess
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ModuleNotFoundError:
    drive = None
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    PROFILE = "colab"
else:
    PROFILE = "local"
    print("google.colab not available; running in local notebook mode.")
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    local_repo_root = next(
        (
            path for path in candidates
            if (path / ".git").exists() and (path / "bench").exists() and (path / "README.md").exists()
        ),
        None,
    )
    if local_repo_root is None:
        raise RuntimeError("Could not locate the repo root from the current local notebook working directory.")
    if WORKSPACE_ROOT == "/content":
        WORKSPACE_ROOT = str(local_repo_root.parent)
    if DRIVE_ROOT.startswith("/content/drive/"):
        DRIVE_ROOT = str((local_repo_root / ".colab-local").resolve())

print(f"IN_COLAB={IN_COLAB}")
print(f"PROFILE={PROFILE}")
print(f"WORKSPACE_ROOT={WORKSPACE_ROOT}")
print(f"DRIVE_ROOT={DRIVE_ROOT}")


In [ ]:
import subprocess
import sys
from pathlib import Path

workspace_root = Path(WORKSPACE_ROOT).expanduser().resolve()
repo_name = Path(REPO_URL.rstrip("/")).name.removesuffix(".git")
repo_root = workspace_root / repo_name

if not IN_COLAB:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    REPO_ROOT = next(
        (
            path for path in candidates
            if (path / ".git").exists() and (path / "bench").exists() and path.name == repo_name
        ),
        None,
    )
    if REPO_ROOT is None:
        raise RuntimeError("Could not find the current repo checkout for local notebook mode.")
else:
    if not (repo_root / ".git").exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(repo_root)],
            check=True,
        )
    REPO_ROOT = repo_root

sys.path.insert(0, str((REPO_ROOT / "bench").resolve()))

from tq_bench.colab import ensure_repo_checkout

if IN_COLAB:
    REPO_ROOT = ensure_repo_checkout(REPO_URL, BRANCH, workspace_root)
print(f"Repo root: {REPO_ROOT}")


In [ ]:
from tq_bench.colab import configure_hf_cache, install_bench_editable

install_bench_editable(REPO_ROOT)
HF_PATHS = configure_hf_cache(DRIVE_ROOT)
print(HF_PATHS)


In [ ]:
from tq_bench.colab import ensure_llama_cpp_checkout, ensure_llama_server, ensure_model_artifacts

DRIVE_ROOT = Path(DRIVE_ROOT).expanduser().resolve()
if IN_COLAB:
    llama_checkout = ensure_llama_cpp_checkout(
        REPO_ROOT,
        llama_repo_url=LLAMA_CPP_REPO_URL,
        branch=LLAMA_CPP_BRANCH,
        commit=LLAMA_CPP_COMMIT or None,
    )
else:
    llama_root = REPO_ROOT / "llama.cpp"
    if not (llama_root / ".git").exists():
        raise RuntimeError("Local notebook mode expects an existing llama.cpp git checkout at REPO_ROOT/llama.cpp")
    llama_checkout = type("LocalCheckout", (), {
        "path": llama_root,
        "repo_url": subprocess.run(["git", "config", "--get", "remote.origin.url"], cwd=str(llama_root), check=True, capture_output=True, text=True).stdout.strip(),
        "commit": subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(llama_root), check=True, capture_output=True, text=True).stdout.strip(),
    })()
artifact = ensure_llama_server(
    REPO_ROOT,
    DRIVE_ROOT,
    force_rebuild=FORCE_REBUILD_SERVER,
    build_dir_name=LLAMA_BUILD_DIR_NAME,
    build_type=LLAMA_BUILD_TYPE,
    cuda_arch_override=LLAMA_CUDA_ARCH_OVERRIDE,
    extra_cmake_args=LLAMA_EXTRA_CMAKE_ARGS,
    build_targets=LLAMA_BUILD_TARGETS,
    build_jobs=LLAMA_BUILD_JOBS,
)

download_requests = [(MODEL_ID, MODEL_QUANT), *EXTRA_MODEL_DOWNLOADS]
downloaded_artifacts = {}
for download_model_id, download_quant in download_requests:
    downloaded_artifacts[(download_model_id, download_quant)] = ensure_model_artifacts(
        REPO_ROOT / "bench" / "configs" / "models.yaml",
        model_id=download_model_id,
        quant=download_quant,
        drive_root=DRIVE_ROOT,
        force_redownload=FORCE_REDOWNLOAD_MODEL,
    )

copied = downloaded_artifacts[(MODEL_ID, MODEL_QUANT)]
RESULTS_RUNS_DIR = DRIVE_ROOT / "results" / "runs"
RESULTS_RUNS_DIR.mkdir(parents=True, exist_ok=True)
SLOT_SAVE_PATH = REPO_ROOT / "kvcache"
SLOT_SAVE_PATH.mkdir(parents=True, exist_ok=True)

print(f"llama.cpp checkout: {llama_checkout.path}")
print(f"llama.cpp remote:   {llama_checkout.repo_url}")
print(f"llama.cpp commit:   {llama_checkout.commit}")
print(f"llama-server: {artifact.binary_path}")
print(f"llama-kv-dump: {artifact.kv_dump_binary_path}")
print(f"CUDA arch:    sm{artifact.cuda_arch}")
print(f"Build dir:    {LLAMA_BUILD_DIR_NAME}")
print(f"Build type:   {LLAMA_BUILD_TYPE}")
print(f"CMake args:   {LLAMA_EXTRA_CMAKE_ARGS}")
print(f"Targets:      {LLAMA_BUILD_TARGETS}")
print(f"Build jobs:   {LLAMA_BUILD_JOBS}")
for (download_model_id, download_quant), artifacts in downloaded_artifacts.items():
    print(f"Downloaded model: {download_model_id} [{download_quant}] -> {artifacts[download_quant]}")
    print(f"mmproj file:      {artifacts.get('mmproj')}")
print(f"Active model file: {copied[MODEL_QUANT]}")
print(f"Results dir:  {RESULTS_RUNS_DIR}")


In [ ]:
import importlib
import os
import subprocess
import sys

sys.path.insert(0, str((REPO_ROOT / "bench").resolve()))

import tq_bench.colab as tq_colab
tq_colab = importlib.reload(tq_colab)
build_run_bench_command = tq_colab.build_run_bench_command
run_command_live = getattr(tq_colab, "run_command_live", None)
if run_command_live is None:
    def run_command_live(cmd, *, cwd=None, env=None, step=None):
        if step:
            print(f"[tq-bench] {step}: {' '.join(cmd)}", flush=True)
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd is not None else None,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        try:
            for line in proc.stdout:
                print(line, end="", flush=True)
        finally:
            proc.stdout.close()
        returncode = proc.wait()
        if returncode != 0:
            raise RuntimeError(f"Command failed with exit code {returncode}: {' '.join(cmd)}")

kv_root = REPO_ROOT / "logs" / "kv_analysis"
kv_root.mkdir(parents=True, exist_ok=True)
SMOKE_KV_RUN_DIRS = []
smoke_before = {path.resolve() for path in kv_root.glob(f"{MODEL_ID}_*") if path.is_dir()}

smoke_benchmarks = SMOKE_BENCHMARKS or ["ai2d"]
SMOKE_OUTPUT = RESULTS_RUNS_DIR / f"{RESULTS_TAG}_smoke.json"
smoke_cmd = build_run_bench_command(
    REPO_ROOT,
    num=1,
    model_id=MODEL_ID,
    model_quant=MODEL_QUANT,
    runtimes=RUNTIMES,
    benchmarks=smoke_benchmarks,
    profile=PROFILE,
    parallel=PARALLEL_OVERRIDE,
    results_dir=RESULTS_RUNS_DIR,
    output_path=SMOKE_OUTPUT,
    slot_save_path=SLOT_SAVE_PATH,
    server_binary_path=artifact.binary_path,
    kv_dump_binary_path=artifact.kv_dump_binary_path,
    extra_args=EXTRA_RUN_BENCH_ARGS,
)
if RUN_KV_DUMP:
    smoke_cmd.append("--kv-dump")
print(" ".join(smoke_cmd))
run_command_live(smoke_cmd, cwd=REPO_ROOT, env=os.environ.copy(), step="smoke")
if RUN_KV_DUMP:
    SMOKE_KV_RUN_DIRS = sorted(
        [path for path in kv_root.glob(f"{MODEL_ID}_*") if path.is_dir() and path.resolve() not in smoke_before],
        key=lambda path: path.stat().st_mtime,
    )
    print("Smoke KV runs:", [str(path) for path in SMOKE_KV_RUN_DIRS])


In [ ]:
import importlib
import os
import subprocess
import sys

sys.path.insert(0, str((REPO_ROOT / "bench").resolve()))

import tq_bench.colab as tq_colab
tq_colab = importlib.reload(tq_colab)
build_run_bench_command = tq_colab.build_run_bench_command
run_command_live = getattr(tq_colab, "run_command_live", None)
if run_command_live is None:
    def run_command_live(cmd, *, cwd=None, env=None, step=None):
        if step:
            print(f"[tq-bench] {step}: {' '.join(cmd)}", flush=True)
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd is not None else None,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        try:
            for line in proc.stdout:
                print(line, end="", flush=True)
        finally:
            proc.stdout.close()
        returncode = proc.wait()
        if returncode != 0:
            raise RuntimeError(f"Command failed with exit code {returncode}: {' '.join(cmd)}")

kv_root = REPO_ROOT / "logs" / "kv_analysis"
kv_root.mkdir(parents=True, exist_ok=True)
MAIN_KV_RUN_DIRS = []
main_before = {path.resolve() for path in kv_root.glob(f"{MODEL_ID}_*") if path.is_dir()}

OUTPUT_PATH = RESULTS_RUNS_DIR / f"{RESULTS_TAG}.json"
resume_path = OUTPUT_PATH if RESUME and OUTPUT_PATH.exists() else None
main_cmd = build_run_bench_command(
    REPO_ROOT,
    num=N_SAMPLES,
    model_id=MODEL_ID,
    model_quant=MODEL_QUANT,
    runtimes=RUNTIMES,
    benchmarks=BENCHMARKS,
    profile=PROFILE,
    parallel=PARALLEL_OVERRIDE,
    results_dir=RESULTS_RUNS_DIR,
    output_path=OUTPUT_PATH,
    resume_path=resume_path,
    slot_save_path=SLOT_SAVE_PATH,
    server_binary_path=artifact.binary_path,
    kv_dump_binary_path=artifact.kv_dump_binary_path,
    extra_args=EXTRA_RUN_BENCH_ARGS,
)
if RUN_KV_DUMP:
    main_cmd.append("--kv-dump")
print(" ".join(main_cmd))
run_command_live(main_cmd, cwd=REPO_ROOT, env=os.environ.copy(), step="main")
if RUN_KV_DUMP:
    MAIN_KV_RUN_DIRS = sorted(
        [path for path in kv_root.glob(f"{MODEL_ID}_*") if path.is_dir() and path.resolve() not in main_before],
        key=lambda path: path.stat().st_mtime,
    )
    print("Main KV runs:", [str(path) for path in MAIN_KV_RUN_DIRS])


In [ ]:
import json

import pandas as pd
from IPython.display import FileLink, FileLinks, Image, Markdown, display


def _display_result_summary(label, output_path):
    output_path = Path(output_path)
    if not output_path.exists():
        print(f"{label} result file not found: {output_path}")
        return None

    print(f"\n=== {label} Results ===")
    with open(output_path) as f:
        payload = json.load(f)

    rows = []
    for rec in payload.get("records", []):
        rows.append(
            {
                "runtime_id": rec.get("runtime_id"),
                "benchmark_id": rec.get("benchmark_id"),
                "status": rec.get("status"),
                "score": rec.get("score"),
                "n_succeeded": rec.get("n_succeeded"),
                "n_samples": rec.get("n_samples"),
                "wall_time_seconds": rec.get("wall_time_seconds"),
            }
        )

    df = pd.DataFrame(rows)
    if not df.empty:
        display(df.sort_values(["runtime_id", "benchmark_id"]).reset_index(drop=True))
    print(json.dumps(payload.get("meta", {}), indent=2))
    print(f"Results file: {output_path}")
    display(FileLink(str(output_path), result_html_prefix="Download JSON: "))
    return payload


def _display_csv_preview(csv_path):
    print(f"Artifact CSV: {csv_path.name}")
    try:
        display(pd.read_csv(csv_path).head())
    except pd.errors.EmptyDataError:
        print("  (empty CSV; no rows/columns to display)")
    display(FileLink(str(csv_path), result_html_prefix="Download CSV: "))


def _display_kv_artifacts(label, run_dirs):
    print(f"\n=== {label} KV Artifacts ===")
    if not run_dirs:
        print("No KV analysis runs found.")
        return

    for run_dir in run_dirs:
        print(f"\nKV analysis root: {run_dir}")
        display(FileLinks(str(run_dir), recursive=True, result_html_prefix="Download artifacts: "))

        dump_dirs = sorted(path for path in (run_dir / "dumps").glob("*") if path.is_dir())
        print("KV dump dirs:", [path.name for path in dump_dirs])
        for dump_dir in dump_dirs:
            meta_path = dump_dir / "meta.json"
            if meta_path.exists():
                print(f"Dump metadata: {dump_dir.name}")
                try:
                    display(pd.json_normalize(json.loads(meta_path.read_text(encoding="utf-8"))))
                except Exception:
                    print(meta_path.read_text(encoding="utf-8")[:1000])
                display(FileLink(str(meta_path), result_html_prefix="Download meta.json: "))

        analysis_dirs = sorted(path for path in (run_dir / "analysis").glob("*") if path.is_dir())
        if not analysis_dirs:
            print("No comparative KV analysis directories found yet.")
        for analysis_dir in analysis_dirs:
            print(f"\n--- analysis: {analysis_dir.name} ---")
            report_path = analysis_dir / "report.md"
            if report_path.exists():
                display(Markdown(report_path.read_text(encoding="utf-8")))
                display(FileLink(str(report_path), result_html_prefix="Download report.md: "))

            csv_paths = sorted(analysis_dir.glob("*.csv"))
            for csv_path in csv_paths:
                _display_csv_preview(csv_path)

            plot_dir = analysis_dir / "plots"
            if plot_dir.exists():
                for plot_path in sorted(plot_dir.glob("*.png")):
                    print(f"Plot: {plot_path.name}")
                    display(Image(filename=str(plot_path)))
                    display(FileLink(str(plot_path), result_html_prefix="Download PNG: "))


SMOKE_PAYLOAD = _display_result_summary("Smoke", SMOKE_OUTPUT)
if RUN_KV_DUMP:
    _display_kv_artifacts("Smoke", globals().get("SMOKE_KV_RUN_DIRS", []))

MAIN_PAYLOAD = _display_result_summary("Main", OUTPUT_PATH)
if RUN_KV_DUMP:
    _display_kv_artifacts("Main", globals().get("MAIN_KV_RUN_DIRS", []))
